In [1]:
"""
This script contains usefull functions used in the PLAID_to_IDOT tools

@author: Jonne Rietdijk
"""

#------------  basic functions --------------#

def log_10_dilutions(max_mM, dilutions=12):
    """ 
    This function creates 10 log doses starting from the highest given dose
    e.g. give 100 returns 100,10,1,0.1,0.01...0.00000000001
    
    Args:
    max_mM (int): highest stock that you want to create 10x dolution from
    dilutions (int): number of dilutions
    
    Returns:
        str: the well name with added zero.
  
    """    
    import math 
    availstocks_mM = []  
    
    maxlog10 = int(math.log(max_mM,10))
    
    for i in range(-dilutions, maxlog10 + 1):
        availstocks_mM.append(pow(10, i))
    availstocks_mM.append(max_mM)
        
    return availstocks_mM




def createplate(size, direction):
    """
    Creates list of well names for 96 or 384 well plates

    Args:
        size (int): 96 or 384
        direction (str): "vert" for vertical placement and "hori" for horizontal placement

    Returns:
        str: the well name with added zero.
    """
    import string

    if size == 96:
        colr = 13
        rowr = 8
    
    if size == 384:
        colr = 25
        rowr = 16
    

    row = list(string.ascii_uppercase[:rowr])
    col = [(f'{i:02d}') for i in range(1, colr, 1)]
    wells = []
      
    if direction == "vert":
        for c in col:
            for r in row:
                wells.append(str(r+c))
        return(wells)
    
    else:
        for r in row:
            for c in col:
                wells.append(str(r+c))
        return(wells)
    

    

def removeleadingzero(x):
    """
    Removes leading zero from a well name (e.g A08 => A8 but B12 => B12)

    Args:
        x (str): well name

    Returns:
        str: the well name with added zero.
    """
    x = x[0] + x[1:3].lstrip("0")
    return x 




def addleadingzero(x):  
    """
    Adds leading zero to a well name (e.g A8 => A08 but B12 => B12)

    Args:
        x (str): well name

    Returns:
        str: the well name with added zero.
    """
    x = x[0] + x[1:3].zfill(2)
    return x 




def get_letter_from_number(number):
    """
    returns a capital letter corresponding to a number (e.g. 1 = A, 2 = B)

    Args:
        x (str): well name

    Returns:
        str: the well name with added zero.
    """
    if not math.isnan(number):
        letter = chr((number) + 64)
    else:
        letter = 'NaN'
    return letter




# --------------- special functions ------------------- #

def stockfinder(concUM, highest_stock_mM, V2_ul, dmso_percmax): 
    import numpy as np
    
    if highest_stock_mM != 0:
        lowest_stock_mM = 0.0000001  
        availstocks_mM = [highest_stock_mM / (10**x) for x in range(int(np.ceil(np.log10(highest_stock_mM / lowest_stock_mM)))+1)]
        
        if sourceplate_type == "S.200":
            MinV1_nl = 30 
        else:
            MinV1_nl = 8

        MaxV1_nl = (dmso_percmax / 100) * (V2_ul * 1000)  
        
        C1_low  = (V2_ul * concUM) / MaxV1_nl
        C1_high = (V2_ul * concUM) / MinV1_nl 
        
        psblstocks = [x for x in availstocks_mM if x >= C1_low and x <= C1_high]
        
        if psblstocks:
            highestStock = max(psblstocks)   # select highest stock for your condition                                            
            return highestStock    
        else:
            raise Exception("not possible to find a suitable stock for requested settings")
    else:
        return 0

def treatmentandcontroldict(cmpdname):
    import re
    
    DMSO, ctrls, blank, trt = ([] for i in range(4))
    catdict = {}
    
    for i in cmpdname:
        if bool(re.search('.*dmso.*', i,re.IGNORECASE)):
            findDMSO = re.findall('.*dmso.*', i, re.IGNORECASE) 
            DMSO.append(i)
            if findDMSO[0] not in catdict:
                catdict[i] = "DMSO" 
            

        elif bool(re.search('.*blank.*', i,re.IGNORECASE)):
            findblank = re.findall('.*blank.*', i, re.IGNORECASE) 
            blank.append(i)
            if findblank[0] not in catdict:
                catdict[i] = "blank" 
            

        elif bool(re.search(r'\[.*?\]', i)):
            findctrl = re.findall(r'\[[a-zA-Z0-9_]{4}?\]', i) 
            ctrls.append(i)
            if findctrl[0] not in catdict:
                catdict[i] = "ctrl" 
            
        else:
            if i not in catdict:
                trt.append(i)
                catdict[i] = "trt" 
    return catdict


def treatmentdict(cmpdname):
    import re
    
    DMSO, ctrls, blank, trt = ([] for i in range(4))
    catdict = {}
    
    for i in cmpdname:
        if bool(re.search('.*dmso.*', i,re.IGNORECASE)):
            findDMSO = re.findall('.*dmso.*', i, re.IGNORECASE) 
            DMSO.append(i)
            if findDMSO[0] not in catdict:
                catdict[i] = "DMSO" 
            

        elif bool(re.search('.*blank.*', i,re.IGNORECASE)):
            findblank = re.findall('.*blank.*', i, re.IGNORECASE) 
            blank.append(i)
            if findblank[0] not in catdict:
                catdict[i] = "blank" 
            
        else:
            if i not in catdict:
                trt.append(i)
                catdict[i] = "trt" 
    return catdict




def normalizeDMSO(df,maxDMSO):
    dfDMSO = df.copy()
    dfDMSO["DMSO_backfill_uL"] = maxDMSO - dfDMSO["Volume [uL]"] # maxDMSO depends on strategy chosen
    dfDMSO["DMSO_backfill_uL"][dfDMSO["DMSO_backfill_uL"] < 0] = 0
    dfDMSO                     = dfDMSO[dfDMSO.DMSO_backfill_uL != 0] 
    dfDMSO.drop(["Volume [uL]"], axis=1)
    dfDMSO["Volume [uL]"]      = dfDMSO["DMSO_backfill_uL"] 
    
    dfDMSO[["Liquid Name","cmpdname","treatment_type"]]   = "DMSO"
    dfDMSO[["highest_stock_mM","stock_conc_mM"]]          = 0
    dfDMSO = dfDMSO[["Target Plate","cmpdname","highest_stock_mM","stock_conc_mM","treatment_type","Target Well","Liquid Name","Volume [uL]"]]
    return dfDMSO
    

def uLfromstock(concUM, stock_conc_mM, working_volume_ul):
    concUM =  (concUM * working_volume_ul) / stock_conc_mM if stock_conc_mM != 0 else 0
    return concUM / 1000

def stockhighestmM(treatment_type,maxmM_treat,maxmM_ctrl):
    if treatment_type == "trt":
        return maxmM_treat
    if treatment_type == "ctrl":
        return maxmM_ctrl
    if treatment_type == "DMSO":
        return 0
    if treatment_type == "blank":
        return 0



def assignsource(df,treatment_type):
    
    import numpy as np
    import math
    import string
    import pandas as pd
    
    seldf                     = df[df['treatment_type']==str(treatment_type)] 
    seldf.stock_conc_mM       = seldf.stock_conc_mM.astype(float)
    seldf["dilutions1in10"]   = np.log10(seldf["highest_stock_mM"].astype(int) / seldf["stock_conc_mM"]).astype(int) + 1
    seldf                     = seldf.groupby(["Liquid Name"])[["cmpdname","stock_conc_mM","dilutions1in10","treatment_type"]].max().reset_index()


    maxdils     = seldf['dilutions1in10'].max() 
    compounds   = np.unique(seldf[['cmpdname']].values).tolist()
    nrcompounds = len(compounds)

# ----------------- source plates ------------------#
    nrsubplates        = math.floor(12 / maxdils)                   
    maxcmpdperplate    = nrsubplates * 8                           
    totalsubplates     = math.ceil(nrcompounds / 8)

    plates             = math.ceil(nrcompounds / maxcmpdperplate)  
    plates             = int(plates)

    # create plates and subplates:
    x = int(math.floor(12/ nrsubplates))
    startcol = [int(f'{i:01d}') for i in range(1, 13, x)] * plates
    startcols = startcol[0:totalsubplates]
    row96 = list(string.ascii_uppercase[:8])
    
    welldict = {}

    for i, cmp in enumerate(compounds):
        for j, dilution in enumerate(range(0,maxdils)):
        
            comp = i+1
            subplate = math.ceil(comp/8)
            subplateindex = subplate - 1
            subplate = treatment_type + "_" + "source" + str(subplate)
    
            cmprow = (list(string.ascii_uppercase[:8]) * totalsubplates)[0:len(compounds)]
            cmprow = str(cmprow[i])

            cmpcol = startcols[subplateindex] 
            cmpcol = cmpcol + j 
            cmpcol = str(cmpcol).zfill(2) 
            well = cmprow+cmpcol
            welldict[i,j] = [well,cmp, j+1, subplate]    


    source = pd.DataFrame.from_dict(welldict,orient='index', columns=['Source Well','cmpdname','dilutions1in10', 'Source Plate'])
    sourceplate = pd.merge(seldf, source,  how='left', left_on=['cmpdname','dilutions1in10'], right_on = ['cmpdname','dilutions1in10'])

    
    return(sourceplate)

def assign_DMSOsource(max_volume, dfvolumes):
    
    
    wellcapacity = int(max_volume *1e06) * 0.8 # wellcapacity based on idot plate (but with 20% safety margin)
    well_state = {"well_number": 0, "current_amount": wellcapacity}
    DMSOwells = createplate(size=96,direction="vert")
    
    sourcewelllist = []

    for volume in dfvolumes:
        remaining = well_state["current_amount"] - volume
        if remaining < 0:
            well_state["well_number"] += 1
            well_state["current_amount"] = wellcapacity
            wellindex = well_state["well_number"]
            sourcewelllist.append(DMSOwells[wellindex])
        else:
            well_state["current_amount"] -= volume
            wellindex = well_state["well_number"]
            sourcewelllist.append(DMSOwells[wellindex])
            
    sourcewells = [*set(sourcewelllist)]      
    return sourcewelllist

def treatmentstodict(conditions):
    import re
    
    DMSO, ctrls, blank, trt = ([] for i in range(4))
    catdict = {}
    
    for i in conditions:
        if bool(re.search('.*dmso.*', i,re.IGNORECASE)):
            findDMSO = re.findall('.*dmso.*', i, re.IGNORECASE) 
            DMSO.append(i)
            if findDMSO[0] not in catdict:
                catdict[i] = "DMSO" 
            

        elif bool(re.search('.*blank.*', i,re.IGNORECASE)):
            findblank = re.findall('.*blank.*', i, re.IGNORECASE) 
            blank.append(i)
            if findblank[0] not in catdict:
                catdict[i] = "blank" 
            

        elif bool(re.search(r'\[.*?\]', i)):
            findctrl = re.findall(r'\[[a-zA-Z0-9_]{4}?\]', i) 
            ctrls.append(i)
            if findctrl[0] not in catdict:
                catdict[i] = "ctrl" 
            
        else:
            if i not in catdict:
                trt.append(i)
                catdict[i] = "trt" 
            
    print("A total of       ",len(conditions),"conditions were found","\n")
    print("blanks           ",len(blank),"blank control(s) are identified in your assay:\n                 ", blank,"\n")
    print("DMSO:            ",len(DMSO), "DMSO control(s) are identified in your assay:\n                 ",DMSO,"\n")
    print("Controls:        ",len(ctrls), "control compounds are identified in your assay:\n                 ", ctrls,"\n")
    print("Treatments:      ",len(trt), "treatments are identified in your assay:\n                 ", trt,"\n")
    
    return catdict


def assignsourcesimple(dfs): # input list of dataframes to be included
    import pandas as pd
    import math

    alldfs             = pd.concat(dfs)
    compounds          = alldfs["Liquid Name"].unique().tolist()
    nrcompounds        = len(compounds)
      
    nrplates           = int(math.ceil(nrcompounds / 96))                                             
    wells              = createplate(size = 96, direction= "vert") * nrplates # generate all wells of the number of plates needed
    wells              = wells[0:nrcompounds]     

    
    welldict = {}
    counter = 96 

    for i, cmpd in enumerate(compounds):
      platenumber = 1
      remaining = counter - i

      if remaining < 0:
          platenumber += 1
      else:
          sourceplate = "sourceplate"+ str(platenumber)
          welldict[i] = [wells[i], cmpd, sourceplate]  
  
    allsourceplates      = pd.DataFrame.from_dict(welldict,orient='index', columns=['Source Well','cmpdname','Source Plate'])

    return allsourceplates

In [2]:
# functions used for validations and to work with idot log files

def extract_compound_info(text):
    try:
        result = pd.Series(text.str.extract(r'^(.+)\[(.+)\]$').values.ravel())
    except ValueError:
        result = pd.Series(['', ''])
    return result


def logtodf(logfile):
    import pandas as pd 
    
    log = pd.read_csv(logfile, sep=',', header=None)
    delimiter = '\t'
    df2 = log[0].str.split(delimiter, expand=True)
    df2 = df2.apply(lambda x: x.str.strip() if x.dtype == "object" else x) # remove all whitespaces
    
    #extract target late names
    plates = df2[df2[0].str.startswith('Target')][0].str.extract(r"\:(.*?)\(")
    plates = plates[0].str.strip().tolist()
    print(plates)
   
    # split log file for each plate
    end = [int(df2.tail(1).index.item())]
    split =  df2.index[df2[0]=="Liquid"].tolist()
    startprot = [i for i in split]
    endprot = startprot[1:] 
    endprot = [i -3 for i in endprot]
    endprot = endprot + end 
    
    # for each target plate, concaternate the rows to a new df
    masterdf = []

    for i, stindex in enumerate(startprot): 
        start = startprot[i] + 1
        end = endprot[i] + 1

        df = df2[start:end]
        df["Target Plate"] = plates[i]
        masterdf.append(df)
    df = pd.concat(masterdf)
    
    df.columns = ["Liquid Name","Source","Target Well","TargetFwd","TargetSide","Drop","Miss","TargetVolume","DosingEnergy","Target Plate"]
    df["Miss"] = pd.to_numeric(df["Miss"])
    df["Drop"] = pd.to_numeric(df["Drop"])

    df[["TargetVolume", "TargetUnit"]] = df["TargetVolume"].str.split(" ", expand=True)
    df["TargetVolume"] = pd.to_numeric(df["TargetVolume"])

    df = df.loc[(df["Target Well"] != "Waste")]
    
    #print("Your output is saved in the output folder")
    #df.to_csv("output/log_file_concatenated.csv", encoding = 'utf-8-sig', index=False) 
    
    return(df)